In [29]:
from IPython.display import Markdown, display
import os
from tqdm import tqdm
from yandex_cloud_ml_sdk import YCloudML
from glob import glob
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
import json
import numpy as np


def printx(string):
    display(Markdown(string))

def create_assistant(model, tools=None):
    kwargs = {}
    if tools and len(tools) > 0:
        kwargs = {"tools": tools}
    return sdk.assistants.create(
        model, ttl_days=1, expiration_policy="since_last_active", **kwargs
    )

folder_id = 'b1gst3c7cskk2big5fqn'
api_key = 'AQVNzzJielnSayrAOlQWlxDMK49OShvzdqtUQdAp'

sdk = YCloudML(folder_id=folder_id, auth=api_key)
model = (
  sdk.models.text_embeddings(model_name="text-search-query", model_version="latest")
)

In [32]:
import json
import pandas as pd
from pathlib import Path


json_dir = '/Users/ogzeus/Downloads/'
json_files = list(Path(json_dir).glob('visual_modified.txt'))
print(json_files)
all_data = []
for json_file in json_files:
    with open(json_file, 'r', encoding='utf-8') as file:
        data = file.readlines()
        if isinstance(data, list):
            all_data.extend(data)
        else:
            all_data.append(data)
df = pd.DataFrame(all_data)

[PosixPath('/Users/ogzeus/Downloads/visual_modified.txt')]


In [34]:
df

,0
0,Будет ли при приёме в институт засчитываться з...
1,Когда пишется заявление на желаемое направлени...
2,Каким внутренним документом регламентируется р...
3,Будет ли при приёме в институт засчитываться з...
4,Когда пишется заявление на желаемое направлени...
...,...
881,Каковы шансы на заселение в общежитие\n
882,Был ли случай что в общежитии освобождались ме...
883,Есть ли статистика сколько приносили оригинало...
884,Когда на сайте появятся обновлённые средние и ...


In [36]:
embeds = []
for i in tqdm(df[0]):
    res = model.run(i)
    embeds.append(res)

  0%|          | 0/886 [00:00<?, ?it/s]

In [39]:
from sklearn.manifold import TSNE
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10
import numpy as np

# Convert embeddings to numpy array
embeddings_array = np.array(embeds)

# Perform T-SNE dimensionality reduction
tsne = TSNE(n_components=2, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings_array)

# Prepare data for Bokeh
source = ColumnDataSource(data=dict(
    x=embeddings_2d[:, 0],
    y=embeddings_2d[:, 1],
    text=df[0].tolist()
))

# Create the plot
output_notebook()
p = figure(width=800, height=600, title="T-SNE Visualization of Text Embeddings")

# Add hover tool
hover = HoverTool(tooltips=[
    ("Text", "@text"),
])
p.add_tools(hover)

# Plot the points
p.circle('x', 'y', size=10, source=source, alpha=0.6)

# Show the plot
show(p)

Loading BokehJS ...

In [ ]:
embeds

In [58]:
from sklearn.cluster import DBSCAN
import numpy as np
import random
from bokeh.plotting import figure, show
from bokeh.palettes import Category20
from bokeh.models import ColumnDataSource, HoverTool, LabelSet
from collections import Counter  # Добавляем импорт

# Улучшенная кластеризация с защитой от шума
clustering = DBSCAN(
    eps=3.6,  # Увеличиваем радиус поиска
    min_samples=10  # Требуем больше точек для формирования кластера
).fit(embeddings_2d)
labels = clustering.labels_

# Постобработка: удаляем маленькие кластеры (считаем их шумом)
min_cluster_size = 15  # Минимальный размер кластера
counts = Counter(labels)
labels = np.array([
    label if (counts[label] >= min_cluster_size and label != -1) 
    else -1 
    for label in labels
])

# Remove unclustered points (label -1)
clustered_mask = labels != -1
embeddings_2d_clustered = embeddings_2d[clustered_mask]
labels_clustered = labels[clustered_mask]
texts_clustered = df[0].iloc[clustered_mask].tolist()

# Get unique clusters and their colors
unique_clusters = sorted(list(set(labels_clustered)))
colors = Category20[20][:len(unique_clusters)]

# Create a dictionary mapping cluster labels to colors
color_dict = dict(zip(unique_clusters, colors))

# Create a list of colors for each point
point_colors = [color_dict[label] for label in labels_clustered]

# Get random message for each cluster
cluster_labels = {}
for cluster_id in unique_clusters:
    cluster_mask = labels_clustered == cluster_id
    cluster_texts = [text for i, text in enumerate(texts_clustered) if labels_clustered[i] == cluster_id]
    cluster_labels[cluster_id] = random.choice(cluster_texts)[:50] + "..." 

# Calculate cluster centers for label positioning
cluster_centers = {}
for cluster_id in unique_clusters:
    cluster_mask = labels_clustered == cluster_id
    cluster_points = embeddings_2d_clustered[cluster_mask]
    center_x = np.mean(cluster_points[:, 0])
    center_y = np.mean(cluster_points[:, 1])
    cluster_centers[cluster_id] = (center_x, center_y)

# Create the plot
p = figure(width=800, height=600, title='Message Clusters with Labels (Noise-Reduced)')

# Add points
source = ColumnDataSource(data=dict(
    x=embeddings_2d_clustered[:, 0],
    y=embeddings_2d_clustered[:, 1],
    text=texts_clustered,
    colors=point_colors
))

p.scatter('x', 'y', color='colors', source=source, alpha=0.6)

# Add hover tool
hover = HoverTool(tooltips=[('Message', '@text')])
p.add_tools(hover)

# Add cluster labels
label_source = ColumnDataSource(data=dict(
    x=[pos[0] for pos in cluster_centers.values()],
    y=[pos[1] for pos in cluster_centers.values()],
    text=[cluster_labels[cluster_id] for cluster_id in cluster_centers.keys()]
))

labels = LabelSet(x='x', y='y', text='text', source=label_source,
                 text_font_size='8pt', text_color='black', 
                 background_fill_color='white', background_fill_alpha=0.7)
p.add_layout(labels)

show(p)

In [ ]:
# программы
# поступление
# места
# списки поступающих
# справки
# общежитие
# бви